# DisasterVQA Filtered Evaluation Notebook (500 Samples)
This notebook evaluates `AbrarAlam/disasterm3-qwen2.5vl7b-mergedFP` on the `QCRI/DisasterVQA` dataset.
It filters out open-ended questions, tests on 500 samples of Binary and Multiple-Choice questions, and evaluates using an automated exact-match script without an LLM judge.

## 1. Environment Setup

In [ ]:
%pip install git+https://github.com/huggingface/transformers
%pip install accelerate datasets evaluate bitsandbytes
%pip install qwen-vl-utils torchvision
%pip install scikit-learn

## 2. Load & Filter Dataset
We filter the dataset to only include `Binary` and `Multiple-Choice` questions, and sample 500 instances.

In [ ]:
from datasets import load_dataset
import random

# Load full dataset
dataset = load_dataset("QCRI/DisasterVQA", split="train")

# Filter for Binary and Multiple-Choice only
filtered_dataset = [item for item in dataset if item.get('question_type', '').lower() in ['binary', 'multiple-choice']]
print(f"Total Binary & MCQ pairs: {len(filtered_dataset)}")

# Randomly sample 500
random.seed(42)
sample_size = min(500, len(filtered_dataset))
sampled_dataset = random.sample(filtered_dataset, sample_size)
print(f"Sampled {sample_size} QA pairs for evaluation.")


## 3. Load Model and Processor

In [ ]:
import torch
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor

model_id = "AbrarAlam/disasterm3-qwen2.5vl7b-mergedFP"

print(f"Loading model: {model_id} ...")
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)

# Use the base instruct processor as Qwen2.5-VL-7B-Instruct is the base
processor = AutoProcessor.from_pretrained("Qwen/Qwen2.5-VL-7B-Instruct")
print("Model and Processor loaded successfully.")


## 4. Run Inference (Batching)
Run generation on the 500 samples.

In [ ]:
from qwen_vl_utils import process_vision_info
from tqdm import tqdm
import gc

# Fix padding side warning
processor.tokenizer.padding_side = 'left'

results = []
batch_size = 1 # Reduced from 4 to 1 to prevent CUDA Out Of Memory

for i in tqdm(range(0, len(sampled_dataset), batch_size)):
    batch = sampled_dataset[i:i+batch_size]
    messages_batch = []
    for item in batch:
        prompt = item['question']
        if item.get('choices'):
            prompt += '\nChoices:'
            for k, v in item['choices'].items():
                prompt += f'\n{k}: {v}'
            prompt += '\nAnswer with the correct option letter.'
        messages = [
            {
                "role": "user",
                "content": [
                    # Limit image resolution to save memory
                    {"type": "image", "image": item['image'], "max_pixels": 512 * 512},
                    {"type": "text", "text": prompt}
                ]
            }
        ]
        messages_batch.append(messages)

    texts = [processor.apply_chat_template(msg, tokenize=False, add_generation_prompt=True) for msg in messages_batch]
    
    image_inputs, video_inputs = process_vision_info(messages_batch)
    inputs = processor(
        text=texts,
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt"
    ).to("cuda")

    with torch.no_grad():
        generated_ids = model.generate(**inputs, max_new_tokens=50)

    generated_ids_trimmed = [
        out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]

    outputs = processor.batch_decode(
        generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )

    for item, out_text in zip(batch, outputs):
        results.append({
            "id": item.get('question_id', 'N/A'),
            "question_type": item.get('question_type', '').capitalize(),
            "instruction": item['question'],
            "ground_truth": item['groundtruth_answer'],
            "model_output": out_text
        })

    del inputs
    torch.cuda.empty_cache()
    gc.collect()

print("Inference completed.")
# Save results to disk to prevent data loss if session restarts
import json
with open('vqa_predictions_500.jsonl', 'w') as f:
    for r in results:
        f.write(json.dumps(r) + '\n')
print("Saved results to vqa_predictions_500.jsonl")


## 5. Exact Match Evaluation
Compare `model_output` with `ground_truth` using string matching.

In [ ]:
import re
import json
import os
def evaluate_exact_match(truth, prediction):
    pred_clean = str(prediction).strip().lower()
    
    if isinstance(truth, list):
        truths = [str(t).strip().lower() for t in truth]
    else:
        truths = [str(truth).strip().lower()]
        
    for t in truths:
        # Use word boundaries to avoid 'no' matching 'not'
        pattern = r'\b' + re.escape(t) + r'\b'
        if re.search(pattern, pred_clean):
            return 1
    return 0

# Load results if not already in memory
if 'results' not in locals():
    if os.path.exists('vqa_predictions_500.jsonl'):
        results = []
        with open('vqa_predictions_500.jsonl', 'r') as f:
            for line in f:
                results.append(json.loads(line))
        print(f"Loaded {len(results)} results from disk.")
    else:
        raise FileNotFoundError("Results not found in memory or on disk. Please run Cell 4 (Inference) first.")

binary_correct = 0
binary_total = 0
mcq_correct = 0
mcq_total = 0

print('--- Debug Output (First 10) ---')
for r in results[:10]:
    print(f"Q-Type: {r['question_type']} | Truth: {r['ground_truth']} | Pred: {repr(r['model_output'])}")
print('-------------------------------')

for r in results:
    truth = r['ground_truth']
    pred = r['model_output']
    is_correct = evaluate_exact_match(truth, pred)
    
    if 'binary' in r['question_type'].lower():
        binary_correct += is_correct
        binary_total += 1
    elif 'multiple-choice' in r['question_type'].lower() or 'mcq' in r['question_type'].lower():
        mcq_correct += is_correct
        mcq_total += 1

binary_accuracy = binary_correct / binary_total if binary_total > 0 else 0
mcq_accuracy = mcq_correct / mcq_total if mcq_total > 0 else 0

print(f"Binary Accuracy: {binary_accuracy:.4f} ({binary_correct}/{binary_total})")
print(f"Multiple-Choice Accuracy: {mcq_accuracy:.4f} ({mcq_correct}/{mcq_total})")


## 6. Generate Markdown Table

In [ ]:
from IPython.display import display, Markdown

md_table = f"""
| Model Architecture | Binary Accuracy (Yes/No) | Multiple-Choice (Acc) | Open-Ended (Acc) |
| :--- | :---: | :---: | :---: |
| **AbrarAlam/disasterm3-qwen2.5vl7b-mergedFP (Ours)** | **{binary_accuracy:.2f}** | **{mcq_accuracy:.2f}** | **N/A (Excluded)** |
| GPT-4.1-mini (Proprietary) | 0.91 | 0.85 | 0.83 |
| Mistral-Small-3.1-24B | 0.90 | 0.81 | 0.75 |
| Qwen-2.5-VL-32B | 0.89 | 0.81 | 0.79 |
| Molmo-7B-D | 0.88 | 0.74 | 0.70 |
| GPT-4o-mini (Proprietary) | 0.87 | 0.78 | 0.78 |
| LLaMA-3.2-11B | 0.87 | 0.80 | 0.75 |
| Pixtral-12B | 0.86 | 0.77 | 0.76 |
"""
display(Markdown(md_table))
print("Copy the above table to your paper or benchmark markdown file.")
